# Búsqueda de ceros — métodos escalares

Plantillas independientes. Cambia f, intervalo o semillas, y df si corresponde. Cada método devuelve la raíz aproximada y el número de iteraciones. No se resuelve la tarea 3.

**Fuente:** `03-ceros, pp. 3–17` y notebook de clase original. Revisión de estudio con IA; usar en evaluación solo si el material asistido está autorizado.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Bisección y posición falsa
Requieren continuidad y un intervalo con cambio de signo (o raíz en un extremo). Bisección tiene una cota de error por ancho; posición falsa conserva el intervalo pero puede estancarse.

In [2]:
def biseccion(f,a,b,tol=1e-8,max_iter=200):
    fa,fb=f(a),f(b)
    if fa==0: return a,0
    if fb==0: return b,0
    if not (a<b and np.sign(fa)!=np.sign(fb)):
        raise ValueError('Se requiere a<b y cambio de signo')
    for i in range(1,max_iter+1):
        c=(a+b)/2
        fc=f(c)
        if fc==0 or (b-a)/2<=tol:
            return c,i
        if np.sign(fa)!=np.sign(fc):
            b,fb=c,fc
        else:
            a,fa=c,fc
    raise RuntimeError('No convergió: revisar tolerancia o intervalo')

def posicion_falsa(f,a,b,tol=1e-8,max_iter=200):
    fa,fb=f(a),f(b)
    if fa==0: return a,0
    if fb==0: return b,0
    if not (a<b and np.sign(fa)!=np.sign(fb)):
        raise ValueError('Se requiere a<b y cambio de signo')
    anterior=None
    for i in range(1,max_iter+1):
        c=b-fb*(b-a)/(fb-fa)
        fc=f(c)
        if fc==0 or (b-a)<=tol or (anterior is not None and abs(c-anterior)<=tol and abs(fc)<=tol):
            return c,i
        if np.sign(fa)!=np.sign(fc):
            b,fb=c,fc
        else:
            a,fa=c,fc
        anterior=c
    raise RuntimeError('No convergió: puede existir estancamiento')

## 2. Newton-Raphson y secante
Newton usa la derivada y una semilla. Secante aproxima la derivada con dos semillas. Ninguno garantiza convergencia global. Se comprueba paso y residuo; eso no garantiza error absoluto en la raíz.

In [3]:
def newton(f,df,x0,tol=1e-8,max_iter=100):
    x=float(x0)
    for i in range(1,max_iter+1):
        fx=f(x)
        if fx==0: return x,i-1
        dfx=df(x)
        if dfx==0: raise ValueError('Derivada nula')
        paso=-fx/dfx
        x=x+paso
        if not np.isfinite(x): raise RuntimeError('Iteración no finita')
        if abs(paso)<=tol and abs(f(x))<=tol: return x,i
    raise RuntimeError('Newton no convergió')

def secante(f,x0,x1,tol=1e-8,max_iter=100):
    f0,f1=f(x0),f(x1)
    for i in range(1,max_iter+1):
        if f1==0: return x1,i-1
        if f1==f0: raise ValueError('Denominador nulo')
        paso=-f1*(x1-x0)/(f1-f0)
        x0,x1=x1,x1+paso
        f0,f1=f1,f(x1)
        if not np.isfinite(x1): raise RuntimeError('Iteración no finita')
        if abs(paso)<=tol and abs(f1)<=tol: return x1,i
    raise RuntimeError('Secante no convergió')

## 3. Ejemplo original: x sin(x)−1
Cambia f, df y las semillas aquí; puedes usar los métodos independientemente.

In [4]:
def f(x): return x*np.sin(x)-1
def df(x): return np.sin(x)+x*np.cos(x)
a,b=0,2
resultados=[biseccion(f,a,b),posicion_falsa(f,a,b),newton(f,df,1.),secante(f,1.,1.5)]
for nombre,(raiz,it) in zip(['Bisección','Posición falsa','Newton','Secante'],resultados):
    print(nombre,raiz,it,'residuo:',abs(f(raiz)))
    assert abs(f(raiz))<1e-7

Bisección 1.114157147705555 28 residuo: 9.490602304040863e-09
Posición falsa 1.1141571408730828 5 residuo: 1.6009416015094757e-12
Newton 1.1141571408719302 4 residuo: 2.220446049250313e-16
Secante 1.1141571408719302 5 residuo: 2.220446049250313e-16


## 4. Buscar varias raíces: localizar intervalos
Un barrido detecta cambios de signo en una malla. Después aplica bisección en cada intervalo. Puede omitir raíces dobles o varias raíces en un mismo tramo; aumentar la resolución no da garantía absoluta.

In [5]:
def intervalos_raices(f,a,b,N=100):
    x=np.linspace(a,b,N+1)
    y=np.array([f(xi) for xi in x])
    exactas=x[y==0]
    intervalos=[]
    for i in range(N):
        if y[i]!=0 and y[i+1]!=0 and np.sign(y[i])!=np.sign(y[i+1]):
            intervalos.append((x[i],x[i+1]))
    return exactas,intervalos

def g(x): return (x-.3)*(x-1.7)
exactas,intervalos=intervalos_raices(g,0,2,23)
raices=list(exactas)+[biseccion(g,a,b)[0] for a,b in intervalos]
print(sorted(raices))
assert np.allclose(sorted(raices),[.3,1.7])

[np.float64(0.29999999896339746), np.float64(1.7000000010366025)]


**Cuidado:** no llames bisección sobre un intervalo de igual signo en sus extremos esperando que detecte todas las raíces. |f(x)| pequeño no implica necesariamente |x−r| pequeño. Para cifras seguras, distingue tolerancia absoluta en x, tolerancia relativa y residuo. Las funciones suponen evaluaciones reales y finitas en los puntos visitados.